In [0]:
from pyspark.sql.functions import *

df = spark.table("workspace.bronze.bronze_orders")

# Cleaning the df (Selecting imp. fields only)
cleaned_df = df.select(
    col("order_id"),
    col("customer_id"),
    col("order_date"),
    col("order_status"),
    col("country_code"),
    col("channel"),
    col("total_amount"),
    col("currency"),
    col("_modified"),
    col("ingestion_ts") 
).filter(col("order_id").isNotNull())

In [0]:
missing_values = ['\\n', '?', '', 'null']

# Adding is_valid flag
# Instead of deleting records, Adding missing values to them
cleaned_df = (
    cleaned_df
    .withColumn(
        "is_valid",
        when(
            col("customer_id").isNull() |
            col("country_code").isNull() |
            col("channel").isNull() |
            lower(trim(col("customer_id"))).isin(missing_values) |
            lower(trim(col("channel"))).isin(missing_values) |
            lower(trim(col("country_code"))).isin(missing_values),
            0
        ).otherwise(1)
    )
    .withColumn(
        "customer_id",
        when(
            col("customer_id").isNull() |
            lower(trim(col("customer_id"))).isin(missing_values),
            None
        ).otherwise(col("customer_id"))
    )
    .withColumn(
        "channel",
        when(
            col("channel").isNull() |
            lower(trim(col("channel"))).isin(missing_values),
            None
        ).otherwise(col("channel"))
    )
    .withColumn(
        "country_code",
        when(
            col("country_code").isNull() |
            lower(trim(col("country_code"))).isin(missing_values),
            None
        ).otherwise(col("country_code"))
    )
)

In [0]:
# Changing the date format
parsed_date = coalesce(
    try_to_date(col("order_date"), "MM/dd/yyyy"),
    try_to_date(col("order_date"), "M/dd/yyyy"),
    try_to_date(col("order_date"), "MM/d/yyyy"),
    try_to_date(col("order_date"), "M/d/yyyy"),

    try_to_date(col("order_date"), "dd/MM/yyyy"),
    try_to_date(col("order_date"), "d/MM/yyyy"),
    try_to_date(col("order_date"), "dd/M/yyyy"),
    try_to_date(col("order_date"), "d/M/yyyy"),

    try_to_date(col("order_date"), "yyyy/MM/dd"),
    try_to_date(col("order_date"), "yyyy/M/dd"),
    try_to_date(col("order_date"), "yyyy/MM/d"),
    try_to_date(col("order_date"), "yyyy/M/d"),

    try_to_date(col("order_date"), "MM-dd-yyyy"),
    try_to_date(col("order_date"), "M-dd-yyyy"),
    try_to_date(col("order_date"), "MM-d-yyyy"),
    try_to_date(col("order_date"), "M-d-yyyy"),

    try_to_date(col("order_date"), "yyyy-MM-dd"),
    try_to_date(col("order_date"), "yyyy-MM-d"),
    try_to_date(col("order_date"), "yyyy-M-dd"),
    try_to_date(col("order_date"), "yyyy-M-d"),

    try_to_date(col("order_date"), "d-MM-yyyy"),
    try_to_date(col("order_date"), "dd-M-yyyy"),
    try_to_date(col("order_date"), "d-M-yyyy")
)

transformed_df = (
    cleaned_df
    .withColumn("parsed_date", parsed_date)
    .withColumn(
        "order_date",
        when(
            col("parsed_date").isNotNull(),
            date_format(col("parsed_date"), "dd-MM-yyyy")
        ).otherwise(col("order_date"))
    )
    .drop("parsed_date")
)

display(transformed_df)

In [0]:
status_map = spark.createDataFrame([
    ("completed", "completed"),
    ("completado", "completed"),
    ("abgeschlossen", "completed"),
    ("पूर्ण", "completed"),
    ("已完成", "completed"),
    
    ("pending", "pending"),
    ("pendiente", "pending"),
    ("ausstehend", "pending"),
    ("लंबित", "pending"),
    ("待处理", "pending"),

    ("cancelled", "cancelled"),
    ("cancelado", "cancelled"),
    ("storniert", "cancelled"),
    ("रद्द", "cancelled"),
    ("已取消", "cancelled"),

    ("shipped", "shipped"),
    ("versandt", "shipped"),
    ("enviado", "shipped"),
    ("भेज दिया", "shipped"),
    ("已发货", "shipped"),
], ["raw_status", "standard_status"])

In [0]:
standardised_df = transformed_df.join(
    status_map,
    col("order_status") == col("raw_status"),
    "left"
)

In [0]:
missing_values = ['\\n', '?', '', 'null']

standardised_df = (
    standardised_df
    .withColumn(
        "is_invalid_status",
        when(col("order_status").isNull() | 
            lower(trim(col("order_status"))).isin(missing_values), 
            1
        ).otherwise(0)
    )
    .withColumn(
        "order_status",
        when(col("order_status").isNull() | 
            lower(trim(col("order_status"))).isin(missing_values),
            None)
        .otherwise(col("order_status"))
    )
    .withColumn(
        "standard_status",
        when(col("standard_status").isNull() | 
            lower(trim(col("standard_status"))).isin(missing_values),
            None)
        .otherwise(col("standard_status"))
    )
    .drop("raw_status")
)

display(standardised_df)

In [0]:
standardised_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.silver_orders")